# Related Artists Spotify Playlist Generator aka RASPG
https://r.isba.co/spotify-playlist-generator-template


## Workflow

![Spotify Playlist Creation Workflow](https://drive.usercontent.google.com/download?id=1ittLJGF67bKfz8o8ohvbBkCeTjQfEShD)

## Account Setup
1. Log in to https://spotify.com
2. Go to https://developer.spotify.com/dashboard/ (log in if needed) to set up a Spotify Developer account

### Explore API endpoints in the Spotify Web API Console  
https://developer.spotify.com/documentation/web-api

## Import required packages

In [1]:
from urllib.parse import urlencode
import requests
import pandas as pd
import json

## How to let your app safely access Spotify on behalf of a user
Your app is your Jupyter Notebook ... for now.  
The user is you but could be somebody else.

## Retrieve the Client ID and Client Secret for your application
- Go to https://developer.spotify.com/dashboard/applications
- Log in and create a new application
- Once the is application created, you will see your Client ID and Client Secret under the Settings button

![Spotify Authorization Code Flow](https://developer.spotify.com/images/documentation/web-api/auth-code-flow.png)

## Set your client_id and client_secret

In [2]:
client_id = '951782aa84c243559e2e7640422391d3'
client_secret = '3d98ea786fbf4f4ea52a2c531ad8969d'

## Get the user authorization code
https://developer.spotify.com/documentation/web-api/concepts/authorization

In [3]:
auth_params= {
    'client_id': client_id,
    'response_type': 'code',
    'redirect_uri': 'https://localhost',
    'scope': 'playlist-modify-public playlist-modify-private'
}
print(f'auth_params: {auth_params}')
auth_params_encoded = urlencode(auth_params)
print(f'auth_params_encoded: {auth_params_encoded}')

auth_url = f'https://accounts.spotify.com/authorize?{auth_params_encoded}'
print(f'Go to this URL to authorize: {auth_url}')

auth_code = input('Enter the authorization code:')

auth_params: {'client_id': '951782aa84c243559e2e7640422391d3', 'response_type': 'code', 'redirect_uri': 'https://localhost', 'scope': 'playlist-modify-public playlist-modify-private'}
auth_params_encoded: client_id=951782aa84c243559e2e7640422391d3&response_type=code&redirect_uri=https%3A%2F%2Flocalhost&scope=playlist-modify-public+playlist-modify-private
Go to this URL to authorize: https://accounts.spotify.com/authorize?client_id=951782aa84c243559e2e7640422391d3&response_type=code&redirect_uri=https%3A%2F%2Flocalhost&scope=playlist-modify-public+playlist-modify-private


Enter the authorization code: AQCgXP56_vcmv9K-qHZeKPP8vPWMUVU-xaoHc0KT8XNhoTiG6CpF-2IWU85WTUfYSeifTdOPPqOO5AdJD0XikyMlFu-ygTQQzexyO9zThB5hq8fErgeAjH-rgyTqilzDI7auxNS3-SYQfqtegNF4YeMZ7NVXxAV4erLBVJ2XssS_wj2x790zJXzH9i2J_hpgvJXJZrlDtEdNCIQhyaWSleorTtiHNg


Go to the generated URL to authorize.

Copy the authorization code (the value after http://localhost/?code=).

In [4]:
auth_code

'AQCgXP56_vcmv9K-qHZeKPP8vPWMUVU-xaoHc0KT8XNhoTiG6CpF-2IWU85WTUfYSeifTdOPPqOO5AdJD0XikyMlFu-ygTQQzexyO9zThB5hq8fErgeAjH-rgyTqilzDI7auxNS3-SYQfqtegNF4YeMZ7NVXxAV4erLBVJ2XssS_wj2x790zJXzH9i2J_hpgvJXJZrlDtEdNCIQhyaWSleorTtiHNg'

## Exchange the authorization code for an access token
https://developer.spotify.com/documentation/web-api/concepts/access-token

In [5]:
token_url = 'https://accounts.spotify.com/api/token'

data = {
    'grant_type': 'authorization_code',
    'code': auth_code,
    'redirect_uri': 'https://localhost',
    'client_id': client_id,
    'client_secret': client_secret
}

response = requests.post(token_url, data=data)
token_response = response.json()

In [6]:
token_response

{'access_token': 'BQAMjC-_qoGjHY-Jow8d1_KnpZ-aQdaBS0YJzWMjSfBnDvb8bMgHKd6A7tdxf3JuibTYHlhV8FzMzMx30DaSA6eM9g7rwKeRiBc4xl4txLOV9J16EzGIvHRoQ6ILtCepDAakdPKxk4Lyjp-MT3Qhk8Lhbk-s_xd7ckOvcFZZa85zMHtqK7m9H6WQK54CAnZc49vA2rJD-dY0j2conRUSVtOT6jn3irMkAAKSjcYB8wqU8cyXGWDbMw',
 'token_type': 'Bearer',
 'expires_in': 3600,
 'refresh_token': 'AQDCkIOv6VG_pEpoMbyoHbO63VDg1jqYuUHNypgkIMxJ4lqPGKkApt5-xhzPe2juOsZJNG8KPUWXO75mkTljEJGYVkbGf6tWgG-gYaoGfjrQ6Vr_D9LF9O9QgFtg8FOfNaM',
 'scope': 'playlist-modify-private playlist-modify-public'}

In [7]:
access_token = token_response['access_token']
access_token

'BQAMjC-_qoGjHY-Jow8d1_KnpZ-aQdaBS0YJzWMjSfBnDvb8bMgHKd6A7tdxf3JuibTYHlhV8FzMzMx30DaSA6eM9g7rwKeRiBc4xl4txLOV9J16EzGIvHRoQ6ILtCepDAakdPKxk4Lyjp-MT3Qhk8Lhbk-s_xd7ckOvcFZZa85zMHtqK7m9H6WQK54CAnZc49vA2rJD-dY0j2conRUSVtOT6jn3irMkAAKSjcYB8wqU8cyXGWDbMw'

### With the access token, you can now access the Spotify API ... until the token expires or is refreshed.

## Set the Authorization header you will pass to every API request to authenticate your app

In [8]:
auth_headers = {
    'Authorization': f'Bearer {access_token}'
}

## Search for an artist and get their artist ID
https://developer.spotify.com/documentation/web-api/reference/search


In [9]:
# Set the artist name to search for
artist_name = 'Porter Robinson'

search_url = 'https://api.spotify.com/v1/search'

params = {
    'q': artist_name,
    'type': 'artist',
    'limit': 1
}

response = requests.get(search_url, headers=auth_headers, params=params)
search_response = response.json()

In [10]:
search_response

{'artists': {'href': 'https://api.spotify.com/v1/search?query=Porter+Robinson&type=artist&offset=0&limit=1',
  'items': [{'external_urls': {'spotify': 'https://open.spotify.com/artist/3dz0NnIZhtKKeXZxLOxCam'},
    'followers': {'href': None, 'total': 1165781},
    'genres': ['complextro',
     'edm',
     'electro house',
     'pop dance',
     'progressive electro house'],
    'href': 'https://api.spotify.com/v1/artists/3dz0NnIZhtKKeXZxLOxCam',
    'id': '3dz0NnIZhtKKeXZxLOxCam',
    'images': [{'height': 640,
      'url': 'https://i.scdn.co/image/ab6761610000e5eb1ac12dcb2cc4fc7c740c5e0c',
      'width': 640},
     {'height': 320,
      'url': 'https://i.scdn.co/image/ab676161000051741ac12dcb2cc4fc7c740c5e0c',
      'width': 320},
     {'height': 160,
      'url': 'https://i.scdn.co/image/ab6761610000f1781ac12dcb2cc4fc7c740c5e0c',
      'width': 160}],
    'name': 'Porter Robinson',
    'popularity': 65,
    'type': 'artist',
    'uri': 'spotify:artist:3dz0NnIZhtKKeXZxLOxCam'}],
  'li

In [11]:
search_response['artists']

{'href': 'https://api.spotify.com/v1/search?query=Porter+Robinson&type=artist&offset=0&limit=1',
 'items': [{'external_urls': {'spotify': 'https://open.spotify.com/artist/3dz0NnIZhtKKeXZxLOxCam'},
   'followers': {'href': None, 'total': 1165781},
   'genres': ['complextro',
    'edm',
    'electro house',
    'pop dance',
    'progressive electro house'],
   'href': 'https://api.spotify.com/v1/artists/3dz0NnIZhtKKeXZxLOxCam',
   'id': '3dz0NnIZhtKKeXZxLOxCam',
   'images': [{'height': 640,
     'url': 'https://i.scdn.co/image/ab6761610000e5eb1ac12dcb2cc4fc7c740c5e0c',
     'width': 640},
    {'height': 320,
     'url': 'https://i.scdn.co/image/ab676161000051741ac12dcb2cc4fc7c740c5e0c',
     'width': 320},
    {'height': 160,
     'url': 'https://i.scdn.co/image/ab6761610000f1781ac12dcb2cc4fc7c740c5e0c',
     'width': 160}],
   'name': 'Porter Robinson',
   'popularity': 65,
   'type': 'artist',
   'uri': 'spotify:artist:3dz0NnIZhtKKeXZxLOxCam'}],
 'limit': 1,
 'next': 'https://api.spot

In [12]:
artist_id = search_response['artists']['items'][0]['id']
artist_id

'3dz0NnIZhtKKeXZxLOxCam'

## Get the artist's top tracks
https://developer.spotify.com/documentation/web-api/reference/get-an-artists-top-tracks

In [13]:
top_tracks_url = f'https://api.spotify.com/v1/artists/{artist_id}/top-tracks'

params = {
    'market': 'US'
}

response = requests.get(top_tracks_url, headers=auth_headers, params=params)
top_tracks_response = response.json()

In [14]:
top_tracks_response

{'tracks': [{'album': {'album_type': 'single',
    'artists': [{'external_urls': {'spotify': 'https://open.spotify.com/artist/3dz0NnIZhtKKeXZxLOxCam'},
      'href': 'https://api.spotify.com/v1/artists/3dz0NnIZhtKKeXZxLOxCam',
      'id': '3dz0NnIZhtKKeXZxLOxCam',
      'name': 'Porter Robinson',
      'type': 'artist',
      'uri': 'spotify:artist:3dz0NnIZhtKKeXZxLOxCam'},
     {'external_urls': {'spotify': 'https://open.spotify.com/artist/4pb4rqWSoGUgxm63xmJ8xc'},
      'href': 'https://api.spotify.com/v1/artists/4pb4rqWSoGUgxm63xmJ8xc',
      'id': '4pb4rqWSoGUgxm63xmJ8xc',
      'name': 'Madeon',
      'type': 'artist',
      'uri': 'spotify:artist:4pb4rqWSoGUgxm63xmJ8xc'}],
    'external_urls': {'spotify': 'https://open.spotify.com/album/2vVfxcvBQAGnEKM3zM0zxw'},
    'href': 'https://api.spotify.com/v1/albums/2vVfxcvBQAGnEKM3zM0zxw',
    'id': '2vVfxcvBQAGnEKM3zM0zxw',
    'images': [{'url': 'https://i.scdn.co/image/ab67616d0000b2731a5eb771120e2ee3f6a44ed7',
      'width': 640,
  

In [15]:
top_tracks = top_tracks_response['tracks']

In [16]:
type(top_tracks)

list

In [17]:
len(top_tracks)

10

## Get the artist's related artists
https://developer.spotify.com/documentation/web-api/reference/get-an-artists-related-artists

In [18]:
related_artists_url = f'https://api.spotify.com/v1/artists/{artist_id}/related-artists'

response = requests.get(related_artists_url, headers=auth_headers)
related_artists_response = response.json()

In [19]:
related_artists = related_artists_response['artists']
related_artists

[{'external_urls': {'spotify': 'https://open.spotify.com/artist/4pb4rqWSoGUgxm63xmJ8xc'},
  'followers': {'href': None, 'total': 651631},
  'genres': ['complextro',
   'edm',
   'electro house',
   'electropop',
   'filter house',
   'nantes indie',
   'pop dance'],
  'href': 'https://api.spotify.com/v1/artists/4pb4rqWSoGUgxm63xmJ8xc',
  'id': '4pb4rqWSoGUgxm63xmJ8xc',
  'images': [{'url': 'https://i.scdn.co/image/ab6761610000e5ebeb3e154881ba6cc3e4d4980d',
    'height': 640,
    'width': 640},
   {'url': 'https://i.scdn.co/image/ab67616100005174eb3e154881ba6cc3e4d4980d',
    'height': 320,
    'width': 320},
   {'url': 'https://i.scdn.co/image/ab6761610000f178eb3e154881ba6cc3e4d4980d',
    'height': 160,
    'width': 160}],
  'name': 'Madeon',
  'popularity': 56,
  'type': 'artist',
  'uri': 'spotify:artist:4pb4rqWSoGUgxm63xmJ8xc'},
 {'external_urls': {'spotify': 'https://open.spotify.com/artist/0jNDKefhfSbLR9sFvcPLHo'},
  'followers': {'href': None, 'total': 696440},
  'genres': ['bro

In [20]:
len(related_artists)

20

## List slicing

```
a[start:stop]  # items start through stop-1
a[start:]      # items start through the rest of the array
a[:stop]       # items from the beginning through stop-1
a[:]           # a copy of the whole array
```

## Get the top tracks for 3 of the related artists
https://developer.spotify.com/documentation/web-api/reference/get-an-artists-top-tracks

In [21]:
for related_artist in related_artists[0:3]:
    # print(f'related_artist: {related_artist}')
    related_artist_id = related_artist['id']
    print(f'Getting top tracks for related artist: {related_artist_id} - {related_artist['name']}')
    top_tracks_url = f'https://api.spotify.com/v1/artists/{related_artist_id}/top-tracks'

    params = {
        'market': 'US'
    }

    response = requests.get(top_tracks_url, headers=auth_headers, params=params)
    top_tracks_response = response.json()

    related_artist_top_tracks = top_tracks_response['tracks']
    
    top_tracks.extend(related_artist_top_tracks)

Getting top tracks for related artist: 4pb4rqWSoGUgxm63xmJ8xc - Madeon
Getting top tracks for related artist: 0jNDKefhfSbLR9sFvcPLHo - San Holo
Getting top tracks for related artist: 2Hchwjfl1DioXcIwbOJkus - Grant


In [22]:
len(top_tracks)

40

In [23]:
top_tracks

[{'album': {'album_type': 'single',
   'artists': [{'external_urls': {'spotify': 'https://open.spotify.com/artist/3dz0NnIZhtKKeXZxLOxCam'},
     'href': 'https://api.spotify.com/v1/artists/3dz0NnIZhtKKeXZxLOxCam',
     'id': '3dz0NnIZhtKKeXZxLOxCam',
     'name': 'Porter Robinson',
     'type': 'artist',
     'uri': 'spotify:artist:3dz0NnIZhtKKeXZxLOxCam'},
    {'external_urls': {'spotify': 'https://open.spotify.com/artist/4pb4rqWSoGUgxm63xmJ8xc'},
     'href': 'https://api.spotify.com/v1/artists/4pb4rqWSoGUgxm63xmJ8xc',
     'id': '4pb4rqWSoGUgxm63xmJ8xc',
     'name': 'Madeon',
     'type': 'artist',
     'uri': 'spotify:artist:4pb4rqWSoGUgxm63xmJ8xc'}],
   'external_urls': {'spotify': 'https://open.spotify.com/album/2vVfxcvBQAGnEKM3zM0zxw'},
   'href': 'https://api.spotify.com/v1/albums/2vVfxcvBQAGnEKM3zM0zxw',
   'id': '2vVfxcvBQAGnEKM3zM0zxw',
   'images': [{'url': 'https://i.scdn.co/image/ab67616d0000b2731a5eb771120e2ee3f6a44ed7',
     'width': 640,
     'height': 640},
    {'url

## Get audio features for the top tracks
https://developer.spotify.com/documentation/web-api/reference/get-audio-features


In [24]:
track_info_list = []

for track in top_tracks:
    track_id = track['id']

    audio_features_url = f'https://api.spotify.com/v1/audio-features/{track_id}'

    response = requests.get(audio_features_url, headers = auth_headers)
    audio_features_response = response.json()
    print(f'audio_features_response: {audio_features_response}')
    track_info = {
        'track_name': track['name'],
        'artist': ', '.join([artist['name'] for artist in track['artists']]),
        'album': track['album']['name'],
        'track_uri': track['uri'],
        'danceability': audio_features_response['danceability'],
        'energy': audio_features_response['energy'],
        'valence': audio_features_response['valence'],
        'popularity': track['popularity']
    }
    track_info_list.append(track_info)

audio_features_response: {'danceability': 0.477, 'energy': 0.796, 'key': 0, 'loudness': -4.499, 'mode': 1, 'speechiness': 0.0369, 'acousticness': 0.00293, 'instrumentalness': 0, 'liveness': 0.184, 'valence': 0.27, 'tempo': 99.859, 'type': 'audio_features', 'id': '2ewEh7LuvToYyGHq7yT8N1', 'uri': 'spotify:track:2ewEh7LuvToYyGHq7yT8N1', 'track_href': 'https://api.spotify.com/v1/tracks/2ewEh7LuvToYyGHq7yT8N1', 'analysis_url': 'https://api.spotify.com/v1/audio-analysis/2ewEh7LuvToYyGHq7yT8N1', 'duration_ms': 218965, 'time_signature': 4}
audio_features_response: {'danceability': 0.464, 'energy': 0.701, 'key': 0, 'loudness': -2.591, 'mode': 0, 'speechiness': 0.0395, 'acousticness': 0.000199, 'instrumentalness': 0, 'liveness': 0.0484, 'valence': 0.277, 'tempo': 147.944, 'type': 'audio_features', 'id': '5iyuwVaMliMdloOzSBxohv', 'uri': 'spotify:track:5iyuwVaMliMdloOzSBxohv', 'track_href': 'https://api.spotify.com/v1/tracks/5iyuwVaMliMdloOzSBxohv', 'analysis_url': 'https://api.spotify.com/v1/audi

In [25]:
track_info_list

[{'track_name': 'Shelter',
  'artist': 'Porter Robinson, Madeon',
  'album': 'Shelter',
  'track_uri': 'spotify:track:2ewEh7LuvToYyGHq7yT8N1',
  'danceability': 0.477,
  'energy': 0.796,
  'valence': 0.27,
  'popularity': 61},
 {'track_name': 'Cheerleader',
  'artist': 'Porter Robinson',
  'album': 'SMILE! :D',
  'track_uri': 'spotify:track:5iyuwVaMliMdloOzSBxohv',
  'danceability': 0.464,
  'energy': 0.701,
  'valence': 0.277,
  'popularity': 58},
 {'track_name': 'Everything Goes On',
  'artist': 'Porter Robinson, League of Legends',
  'album': 'Everything Goes On',
  'track_uri': 'spotify:track:3WBRfkOozHEsG0hbrBzwlm',
  'danceability': 0.55,
  'energy': 0.648,
  'valence': 0.323,
  'popularity': 60},
 {'track_name': 'Goodbye To A World',
  'artist': 'Porter Robinson',
  'album': 'Worlds',
  'track_uri': 'spotify:track:786ymAh5BmHoIpvjyrvjXk',
  'danceability': 0.444,
  'energy': 0.35,
  'valence': 0.0973,
  'popularity': 58},
 {'track_name': 'Knock Yourself Out XD',
  'artist': 'Por


## Save the top tracks info to a dataframe for analysis

In [26]:
df_top_tracks = pd.DataFrame(track_info_list)

In [27]:
df_top_tracks.head()

,track_name,artist,album,track_uri,danceability,energy,valence,popularity
0,Shelter,"Porter Robinson, Madeon",Shelter,spotify:track:2ewEh7LuvToYyGHq7yT8N1,0.477,0.796,0.2700,61
1,Cheerleader,Porter Robinson,SMILE! :D,spotify:track:5iyuwVaMliMdloOzSBxohv,0.464,0.701,0.2770,58
2,Everything Goes On,"Porter Robinson, League of Legends",Everything Goes On,spotify:track:3WBRfkOozHEsG0hbrBzwlm,0.550,0.648,0.3230,60
3,Goodbye To A World,Porter Robinson,Worlds,spotify:track:786ymAh5BmHoIpvjyrvjXk,0.444,0.350,0.0973,58
4,Knock Yourself Out XD,Porter Robinson,SMILE! :D,spotify:track:6yfY7OdAz4VEOjPUOP3yzN,0.554,0.741,0.3320,56


In [28]:
'''
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df_top_tracks)
'''

'\nfrom google.colab import sheets\nsheet = sheets.InteractiveSheet(df=df_top_tracks)\n'

# Filter tracks based on audio features
Select the audio features and thresholds based on your preferences.

In [29]:
df_playlist_tracks = df_top_tracks.query('danceability < 0.7 and valence < 0.7')
df_playlist_tracks

,track_name,artist,album,track_uri,danceability,energy,valence,popularity
0,Shelter,"Porter Robinson, Madeon",Shelter,spotify:track:2ewEh7LuvToYyGHq7yT8N1,0.477,0.796,0.2700,61
1,Cheerleader,Porter Robinson,SMILE! :D,spotify:track:5iyuwVaMliMdloOzSBxohv,0.464,0.701,0.2770,58
2,Everything Goes On,"Porter Robinson, League of Legends",Everything Goes On,spotify:track:3WBRfkOozHEsG0hbrBzwlm,0.550,0.648,0.3230,60
3,Goodbye To A World,Porter Robinson,Worlds,spotify:track:786ymAh5BmHoIpvjyrvjXk,0.444,0.350,0.0973,58
4,Knock Yourself Out XD,Porter Robinson,SMILE! :D,spotify:track:6yfY7OdAz4VEOjPUOP3yzN,0.554,0.741,0.3320,56
5,Is There Really No Happiness?,Porter Robinson,SMILE! :D,spotify:track:3eE6IderHf7lnqOorqiNVK,0.608,0.838,0.6310,57
6,Russian Roulette,Porter Robinson,SMILE! :D,spotify:track:3RdUgst4llmyJkVtNaqHGv,0.550,0.780,0.2200,55
7,Sad Machine,Porter Robinson,Worlds,spotify:track:1JY6B9ILvmRla2IKKRZvnH,0.470,0.706,0.3270,55
8,Look at the Sky,Porter Robinson,Nurture,spotify:track:5lXNcc8QeM9KpAWNHAL0iS,0.568,0.564,0.3850,55
10,Shelter,"Porter Robinson, Madeon",Shelter,spotify:track:2ewEh7LuvToYyGHq7yT8N1,0.477,0.796,0.2700,61


In [30]:
len(df_playlist_tracks)

35

# Save the playlist tracks to a CSV file
You can analyze the file in Excel, a SQL database, visualization tool, or any other BI tool.

In [31]:
df_playlist_tracks.to_csv('playlist_tracks.csv', index=False)

## Get your Spotify user ID
The user ID is required to specify which Spotify account the new playlist will be created in.

Go to https://www.spotify.com/us/account/profile/


In [32]:
user_id = 'm4tthewcc'

## Create a playlist
https://developer.spotify.com/documentation/web-api/reference/create-playlist

In [33]:
playlist_url = f'https://api.spotify.com/v1/users/{user_id}/playlists'

playlist_name = f'{artist_name} Mixtape'
playlist_description = f'Python-generated playlist using artists related to {artist_name}'

data = json.dumps({
    'name': playlist_name,
    'description': playlist_description,
    'public': False
})

response = requests.post(playlist_url, headers = auth_headers, data = data)
playlist_response = response.json()

In [34]:
playlist_response

{'collaborative': False,
 'description': 'Python-generated playlist using artists related to Porter Robinson',
 'external_urls': {'spotify': 'https://open.spotify.com/playlist/2rFArqDobDNg2aSe18lnHS'},
 'followers': {'href': None, 'total': 0},
 'href': 'https://api.spotify.com/v1/playlists/2rFArqDobDNg2aSe18lnHS',
 'id': '2rFArqDobDNg2aSe18lnHS',
 'images': [],
 'primary_color': None,
 'name': 'Porter Robinson Mixtape',
 'type': 'playlist',
 'uri': 'spotify:playlist:2rFArqDobDNg2aSe18lnHS',
 'owner': {'href': 'https://api.spotify.com/v1/users/m4tthewcc',
  'id': 'm4tthewcc',
  'type': 'user',
  'uri': 'spotify:user:m4tthewcc',
  'display_name': None,
  'external_urls': {'spotify': 'https://open.spotify.com/user/m4tthewcc'}},
 'public': False,
 'snapshot_id': 'AAABk7FRHkm16q9TOEwYE7RmNvt/ZhMa',
 'tracks': {'limit': 100,
  'next': None,
  'offset': 0,
  'previous': None,
  'href': 'https://api.spotify.com/v1/playlists/2rFArqDobDNg2aSe18lnHS/tracks',
  'total': 0,
  'items': []}}

## Get the newly generated playlist ID

In [35]:
playlist_id = playlist_response['id']
playlist_id

'2rFArqDobDNg2aSe18lnHS'

## Get the track URIs (Uniform Resource Indicator) to then add to the playlist

In [36]:
df_playlist_tracks

,track_name,artist,album,track_uri,danceability,energy,valence,popularity
0,Shelter,"Porter Robinson, Madeon",Shelter,spotify:track:2ewEh7LuvToYyGHq7yT8N1,0.477,0.796,0.2700,61
1,Cheerleader,Porter Robinson,SMILE! :D,spotify:track:5iyuwVaMliMdloOzSBxohv,0.464,0.701,0.2770,58
2,Everything Goes On,"Porter Robinson, League of Legends",Everything Goes On,spotify:track:3WBRfkOozHEsG0hbrBzwlm,0.550,0.648,0.3230,60
3,Goodbye To A World,Porter Robinson,Worlds,spotify:track:786ymAh5BmHoIpvjyrvjXk,0.444,0.350,0.0973,58
4,Knock Yourself Out XD,Porter Robinson,SMILE! :D,spotify:track:6yfY7OdAz4VEOjPUOP3yzN,0.554,0.741,0.3320,56
5,Is There Really No Happiness?,Porter Robinson,SMILE! :D,spotify:track:3eE6IderHf7lnqOorqiNVK,0.608,0.838,0.6310,57
6,Russian Roulette,Porter Robinson,SMILE! :D,spotify:track:3RdUgst4llmyJkVtNaqHGv,0.550,0.780,0.2200,55
7,Sad Machine,Porter Robinson,Worlds,spotify:track:1JY6B9ILvmRla2IKKRZvnH,0.470,0.706,0.3270,55
8,Look at the Sky,Porter Robinson,Nurture,spotify:track:5lXNcc8QeM9KpAWNHAL0iS,0.568,0.564,0.3850,55
10,Shelter,"Porter Robinson, Madeon",Shelter,spotify:track:2ewEh7LuvToYyGHq7yT8N1,0.477,0.796,0.2700,61


In [37]:
track_uris = []

for index, row in df_playlist_tracks.iterrows():
    track_uri = row['track_uri']
    print(f'track_uri: {track_uri}')
    track_uris.append(track_uri)

track_uri: spotify:track:2ewEh7LuvToYyGHq7yT8N1
track_uri: spotify:track:5iyuwVaMliMdloOzSBxohv
track_uri: spotify:track:3WBRfkOozHEsG0hbrBzwlm
track_uri: spotify:track:786ymAh5BmHoIpvjyrvjXk
track_uri: spotify:track:6yfY7OdAz4VEOjPUOP3yzN
track_uri: spotify:track:3eE6IderHf7lnqOorqiNVK
track_uri: spotify:track:3RdUgst4llmyJkVtNaqHGv
track_uri: spotify:track:1JY6B9ILvmRla2IKKRZvnH
track_uri: spotify:track:5lXNcc8QeM9KpAWNHAL0iS
track_uri: spotify:track:2ewEh7LuvToYyGHq7yT8N1
track_uri: spotify:track:5wM6LOw2U6XeIFHfsgI6wU
track_uri: spotify:track:4PTPZeJlK1rYlYr6bf11hk
track_uri: spotify:track:4ATmY1hv93ehw77LrIdbEh
track_uri: spotify:track:1IzM5hoLr4bUq57FOrWXQ2
track_uri: spotify:track:3ArdPRbscsYB2uI70AzpuG
track_uri: spotify:track:36HA2woC1glnLa4vqWY3IC
track_uri: spotify:track:3rsq3qtqNBId48c0IVraTS
track_uri: spotify:track:6jq6rcOikCZAmjliAgAmfT
track_uri: spotify:track:2AHd1SRo5rAFvJOCEwLyTi
track_uri: spotify:track:7LAFTet7LRO1ydBaqNRgr9
track_uri: spotify:track:7kZVirTc3V6d4YW

In [38]:
track_uris

['spotify:track:2ewEh7LuvToYyGHq7yT8N1',
 'spotify:track:5iyuwVaMliMdloOzSBxohv',
 'spotify:track:3WBRfkOozHEsG0hbrBzwlm',
 'spotify:track:786ymAh5BmHoIpvjyrvjXk',
 'spotify:track:6yfY7OdAz4VEOjPUOP3yzN',
 'spotify:track:3eE6IderHf7lnqOorqiNVK',
 'spotify:track:3RdUgst4llmyJkVtNaqHGv',
 'spotify:track:1JY6B9ILvmRla2IKKRZvnH',
 'spotify:track:5lXNcc8QeM9KpAWNHAL0iS',
 'spotify:track:2ewEh7LuvToYyGHq7yT8N1',
 'spotify:track:5wM6LOw2U6XeIFHfsgI6wU',
 'spotify:track:4PTPZeJlK1rYlYr6bf11hk',
 'spotify:track:4ATmY1hv93ehw77LrIdbEh',
 'spotify:track:1IzM5hoLr4bUq57FOrWXQ2',
 'spotify:track:3ArdPRbscsYB2uI70AzpuG',
 'spotify:track:36HA2woC1glnLa4vqWY3IC',
 'spotify:track:3rsq3qtqNBId48c0IVraTS',
 'spotify:track:6jq6rcOikCZAmjliAgAmfT',
 'spotify:track:2AHd1SRo5rAFvJOCEwLyTi',
 'spotify:track:7LAFTet7LRO1ydBaqNRgr9',
 'spotify:track:7kZVirTc3V6d4YWtu6RaAu',
 'spotify:track:5BzLCBy61N1a6la0MVtueV',
 'spotify:track:6VpRQDOM7nQ6viuVUSXWGn',
 'spotify:track:4nBTt4n122lScgQwHzKVCV',
 'spotify:track:

In [39]:
track_uris = df_playlist_tracks['track_uri'].tolist()

In [40]:
track_uris

['spotify:track:2ewEh7LuvToYyGHq7yT8N1',
 'spotify:track:5iyuwVaMliMdloOzSBxohv',
 'spotify:track:3WBRfkOozHEsG0hbrBzwlm',
 'spotify:track:786ymAh5BmHoIpvjyrvjXk',
 'spotify:track:6yfY7OdAz4VEOjPUOP3yzN',
 'spotify:track:3eE6IderHf7lnqOorqiNVK',
 'spotify:track:3RdUgst4llmyJkVtNaqHGv',
 'spotify:track:1JY6B9ILvmRla2IKKRZvnH',
 'spotify:track:5lXNcc8QeM9KpAWNHAL0iS',
 'spotify:track:2ewEh7LuvToYyGHq7yT8N1',
 'spotify:track:5wM6LOw2U6XeIFHfsgI6wU',
 'spotify:track:4PTPZeJlK1rYlYr6bf11hk',
 'spotify:track:4ATmY1hv93ehw77LrIdbEh',
 'spotify:track:1IzM5hoLr4bUq57FOrWXQ2',
 'spotify:track:3ArdPRbscsYB2uI70AzpuG',
 'spotify:track:36HA2woC1glnLa4vqWY3IC',
 'spotify:track:3rsq3qtqNBId48c0IVraTS',
 'spotify:track:6jq6rcOikCZAmjliAgAmfT',
 'spotify:track:2AHd1SRo5rAFvJOCEwLyTi',
 'spotify:track:7LAFTet7LRO1ydBaqNRgr9',
 'spotify:track:7kZVirTc3V6d4YWtu6RaAu',
 'spotify:track:5BzLCBy61N1a6la0MVtueV',
 'spotify:track:6VpRQDOM7nQ6viuVUSXWGn',
 'spotify:track:4nBTt4n122lScgQwHzKVCV',
 'spotify:track:

## Add tracks to the playlist
https://developer.spotify.com/documentation/web-api/reference/add-tracks-to-playlist

In [41]:
playlist_tracks_url = f'https://api.spotify.com/v1/playlists/{playlist_id}/tracks'

data = json.dumps({
    'uris': track_uris
})

response = requests.post(playlist_tracks_url, headers=auth_headers, data=data)
playlist_tracks_response = response.json()

In [42]:
playlist_tracks_response

{'snapshot_id': 'AAAAAuvKEg9uhCuYsPP3c1DGMaOX6PPC'}

## Create another playlist! 🎶
Update the artist_name value.

If your access token is still valid, you can "Run cell and below" starting from the "Search for an artist and get their artist ID" cell.

